# Stripe India Job Scraper
## India Jobs Scraper (v2.1 - March 2026)
**Source:** boards-api.greenhouse.io/v1/boards/stripe/jobs

**ATS Detection:** Automatic API + Selenium fallback

In [1]:
!pip install selenium webdriver-manager pandas openpyxl requests beautifulsoup4 lxml playwright -q


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import sys, time, random, re
from pathlib import Path

# Add scripts dir to path so we can import scraper_utils
SCRIPTS_DIR = Path.home() / "Job_Scrapers" / "All_Scripts"
sys.path.insert(0, str(SCRIPTS_DIR))

from scraper_utils import *
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta

# ── LOCATION CONFIG ──────────────────────────────────────────────────────────
# Change to "" to scrape globally (all countries).
# The matching pipeline's pre-filter handles India-specific narrowing.
# Set to "India" here only if you want to reduce volume at scrape time.
LOCATION_FILTER = ""
COUNTRY_CODE   = ""  # e.g. "in" for SmartRecruiters country= param; "" = all
# ─────────────────────────────────────────────────────────────────────────────

print("Imports loaded. Date:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print(f"Location filter: '{LOCATION_FILTER}' (empty = broad/global scraping)")


scraper_utils.py loaded successfully
Schema: 25 columns
Skills DB: 79 skills
Imports loaded. Date: 2026-04-01 00:47:53
Location filter: '' (empty = broad/global scraping)


In [3]:
COMPANY = "Stripe"
OUTPUT_DIR = get_output_dir(COMPANY)
print(f"Output directory: {OUTPUT_DIR}")


Output directory: /Users/incognito/Job_Scrapers/All_CSV_Outputs/Stripe/Outputs/2026_04_01


In [4]:
print("=" * 60)
print("STRIPE INDIA JOB SCRAPER")
print("ATS: Greenhouse (boards-api.greenhouse.io/stripe)")
print("=" * 60)

stripe_jobs = scrape_greenhouse(
    board_token="stripe",
    company_name="Stripe",
    industry="Financial Technology / Payments",
    location_filter=LOCATION_FILTER,
    max_jobs=500
)

# Greenhouse may list jobs with broad regions — also check common Indian cities
if len(stripe_jobs) < 3:
    print("\n  Checking individual Indian city offices...")
    import requests
    from bs4 import BeautifulSoup

    session = get_session()
    resp = session.get(
        "https://boards-api.greenhouse.io/v1/boards/stripe/jobs",
        params={"content": "true"},
        timeout=30
    )
    if resp.status_code == 200:
        all_jobs = resp.json().get("jobs", [])
        india_keywords = ["india", "bengaluru", "bangalore", "mumbai", "pune",
                          "chennai", "hyderabad", "delhi", "gurugram", "noida", "apac"]
        for posting in all_jobs:
            loc = posting.get("location", {}).get("name", "").lower()
            offices = posting.get("offices", [])
            office_names = " ".join([o.get("location", "") + " " + o.get("name", "")
                                     for o in offices]).lower()
            if any(k in loc or k in office_names for k in india_keywords):
                if posting.get("id") not in [j["job_id"] for j in stripe_jobs]:
                    dept = (posting.get("departments", [{}])[0].get("name", "")
                            if posting.get("departments") else "")
                    jd_text = html_to_text(posting.get("content", ""))
                    stripe_jobs.append({
                        "job_id": str(posting.get("id", "")),
                        "title": posting.get("title", ""),
                        "company_name": "Stripe",
                        "job_url": posting.get("absolute_url", ""),
                        "business_unit": dept,
                        "raw_jd_text": jd_text,
                        "location_city": posting.get("location", {}).get("name", "India").split(",")[0].strip(),
                        "location_country": "India",
                        "industry": "Financial Technology / Payments",
                        "date_posted": (posting.get("updated_at", "")[:10]
                                        or datetime.now().strftime("%Y-%m-%d")),
                        "is_active": True,
                        "salary_currency": "INR",
                        "source_platform": "Greenhouse",
                    })

print(f"Total Stripe India jobs: {len(stripe_jobs)}")


STRIPE INDIA JOB SCRAPER
ATS: Greenhouse (boards-api.greenhouse.io/stripe)
  Scraping Stripe via Greenhouse API (board: stripe)


  Total jobs on Greenhouse board: 509
  Mode: BROAD — using all 509 jobs (no location filter)
  Total Stripe India jobs: 500
Total Stripe India jobs: 500


In [5]:
df_stripe = save_results(stripe_jobs, "Stripe", OUTPUT_DIR)
if df_stripe is not None:
    print(f"\nSample jobs:")
    cols = ["title","location_city","seniority_level","business_unit","job_url"]
    cols = [c for c in cols if c in df_stripe.columns]
    print(df_stripe[cols].head(10).to_string())


  [OK] Saved 500 jobs -> Stripe_jobs_2026-04-01.csv
       Seniority: {'junior': 490, 'lead': 8, 'mid': 1, 'senior': 1}
       Work mode: {'onsite': 444, 'remote': 43, 'hybrid': 13}
       Has JD text: 500/500
       Has job URL: 500/500
       Has business unit: 500/500

Sample jobs:
                                                    title  location_city seniority_level                              business_unit                                        job_url
0                             Account Executive, AI Sales  San Francisco          junior  1175 Enterprise - Account Executives (NA)  https://stripe.com/jobs/search?gh_jid=7532733
1                             Account Executive, AI Sales  San Francisco          junior           1650 AI GTM Strategy & Solutions  https://stripe.com/jobs/search?gh_jid=7546284
2       Account Executive, Enterprise (Existing Business)  San Francisco          junior  1175 Enterprise - Account Executives (NA)  https://stripe.com/jobs/search?gh_jid=675527